In [1]:
import os
from pathlib import Path

import pandas as pd

In [2]:
data_path = Path(os.environ["DATA_PATH"])
ghsl_path = Path(os.environ["GHSL_PATH"])
out_path = data_path / "generated"

results_path = Path("./results")
results_path.mkdir(exist_ok=True)

In [3]:
ZONE = "MEX+Monterrey"

In [4]:
zones = []
for path in (out_path / "small" / "emissions").glob("*.csv"):
    if not path.stem.startswith("MEX"):
        continue
    zones.append(path.stem.split("+")[-1])

In [5]:
COLUMN_NAME_MAP = {
    "croplands": "Cultivos",
    "flooded": "Inundado",
    "forests_primary": "Bosques primarios",
    "forests_secondary": "Bosques secundarios",
    "forests_mangroves": "Manglares",
    "grasslands": "Pastizales",
    "other": "Otro",
    "pastures": "Pastizales\np/ganado",
    "settlements": "Asentamientos",
    "shrublands": "Matorrales",
    "wetlands": "Humedales",
}

# Área

In [6]:
def generate_area_table(city: str) -> pd.DataFrame:
    return (
        pd.read_csv(out_path / "small" / "area" / "table_merged" / f"MEX+{city}.csv")
        .set_index("label")
        .T.reset_index(names="year")
        .assign(year=lambda df: df["year"].astype(int) + 2000)
        .query("2000 <= year <= 2020")
        .assign(year=lambda df: df["year"].astype(str))
        .set_index("year")
        .divide(100)
        .rename(columns=COLUMN_NAME_MAP)
    )


area_dir = results_path / "area"
area_dir.mkdir(exist_ok=True)

for zone in zones:
    generate_area_table(zone).to_csv(area_dir / f"{zone}.csv")

# Emisiones totales

In [7]:
def generate_emissions_table(city: str) -> pd.DataFrame:
    df_out = pd.read_csv(out_path / "small" / "emissions" / f"MEX+{city}.csv").query(
        "time_period < 21",
    )
    wanted_cols = [
        "emission_co2e_subsector_total_frst",
        "emission_co2e_subsector_total_lndu",
        "emission_co2e_co2_soil_soc_mineral_soils",
        "emission_co2e_n2o_soil_mineral_soils",
        "emission_co2e_n2o_soil_organic_soils",
        "time_period",
    ]

    name_map = {
        "emission_co2e_subsector_total_frst": "Bosques",
        "emission_co2e_subsector_total_lndu": "Transiciones de uso de suelo",
        "emission_co2e_co2_soil_soc_mineral_soils": "Mineralización del suelo",
        "emission_co2e_n2o_soil_mineral_soils": "Mineralización del suelo\n(equivalente N2O)",
        "emission_co2e_n2o_soil_organic_soils": "Suelo orgánico\n(equivalente N2O)",
    }

    return (
        df_out[wanted_cols]
        .rename(columns=name_map)
        .assign(time_period=lambda df: df["time_period"] + 2000)
        .set_index("time_period")
    )


emissions_dir = results_path / "emissions"
emissions_dir.mkdir(exist_ok=True)

for zone in zones:
    generate_emissions_table(zone).to_csv(emissions_dir / f"{zone}.csv")

# Emisiones transición

In [12]:
def prettify_column(c: str, pretty_labels: bool) -> str:
    if c == "year":
        return c

    c = c.replace("emission_co2e_co2_lndu_conversion_", "")
    start, end = c.split("_to_")
    if pretty_labels:
        return f"{COLUMN_NAME_MAP[start]}-{COLUMN_NAME_MAP[end]}"
    return f"{start}-{end}"


def generate_emissions_by_transition_table(
    city: str,
    *,
    start: str | None = None,
    end: str | None = None,
) -> pd.DataFrame:
    df_transitions = (
        pd.read_csv(out_path / "small" / "transitions_emissions" / f"MEX+{city}.csv")
        .assign(time_period=lambda df: df["time_period"] + 2000)
        .query("2000 <= time_period <= 2020")
        .assign(time_period=lambda df: df["time_period"].astype(str))
        .set_index("time_period")
    )
    df_transitions.columns = [
        prettify_column(c, pretty_labels=True) for c in df_transitions.columns
    ]

    if start is not None:
        return (
            df_transitions[[c for c in df_transitions.columns if c.startswith(start)]]
            .sum(axis=1)
            .multiply(1000)
            .rename("Emisiones (miles de toneladas CO2)")
            .to_frame()
        )

    if end is not None:
        return (
            df_transitions[[c for c in df_transitions.columns if c.endswith(end)]]
            .sum(axis=1)
            .multiply(1000)
            .rename("Emisiones (miles de toneladas CO2)")
            .to_frame()
        )

    if start is not None and end is not None:
        err = "Cannot specify both start and end"
        raise ValueError(err)

    err = ValueError("Must specify start or end")
    raise err


emissions_from_forests_dir = results_path / "emissions_from_forests"
emissions_from_forests_dir.mkdir(exist_ok=True)
for zone in zones:
    generate_emissions_by_transition_table(zone, start="Bosques").to_csv(
        emissions_from_forests_dir / f"{zone}.csv",
    )

emissions_to_cities_dir = results_path / "emissions_to_cities"
emissions_to_cities_dir.mkdir(exist_ok=True)
for zone in zones:
    generate_emissions_by_transition_table(zone, end="Asentamientos").to_csv(
        emissions_to_cities_dir / f"{zone}.csv",
    )